In [28]:
import datasets
from tinkerbell.renderer import Renderer

In [29]:
dataset = datasets.load_dataset("HuggingFaceH4/no_robots")


In [30]:
dataset["train"].shape

(9500, 4)

In [31]:
from typing import cast

dataset = cast(datasets.DatasetDict, dataset)
train_dataset = dataset["train"]
test_dataset = dataset["test"]
# dataset = dataset.shuffle(seed=0)
# test_ds = dataset.take(512)
# train_ds = dataset.skip(512)

In [32]:
train_dataset.shape

(9500, 4)

In [33]:
test_dataset.shape

(500, 4)

In [46]:
data = test_dataset

In [47]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
renderer = Renderer(tokenizer)

In [52]:
datums = renderer.build_chat_examples(data["messages"])

In [53]:
datums

[Datum(model_input=ModelInput(input_ids=TensorData(data=[151644, 8948, 198, 32, 3667, 374, 264, 6236, 6331, 879, 11253, 4755, 448, 21669, 78709, 13, 151645, 198, 151644, 872, 198, 9064, 1521, 17931, 81216, 30, 151645, 198, 151644, 77091, 198, 92964, 374, 220, 19, 15, 15, 15, 1635, 2310, 10270, 327, 3955, 374, 1380, 432, 572, 1156, 6088, 151645, 198, 151644, 872, 198, 9064, 572, 14074, 17931, 35492, 30, 151645, 198, 151644, 77091, 198, 13218, 26423, 572, 279, 1156, 311, 912, 14074, 14, 1249, 1281, 862, 17931, 10876, 438, 39944, 151645, 198, 151644, 872, 198, 3838, 525, 1045, 1661, 71413, 429, 990, 17931, 30, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 71284, 11, 44815, 11, 8279, 11, 323, 19145, 14, 92964, 374, 2244, 311, 22544, 151645, 198], dtype='int64', shape=[120]), attention_mask=TensorData(data=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [54]:
first_datum = datums[0]

In [61]:
from torch.utils.data import Dataset, DataLoader
from tinkerbell.types.datum import Datum

class DatumDataset(Dataset):
    """PyTorch Dataset wrapper for list of Datum objects."""
    
    def __init__(self, datums):
        self.datums = datums
    
    def __len__(self):
        return len(self.datums)
    
    def __getitem__(self, idx):
        return self.datums[idx]

# Create dataset and dataloader
datum_dataset = DatumDataset(datums)
dataloader = DataLoader(datum_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: x)

# Test it out - returns list of Datum objects
for batch in dataloader:
    print(f"Batch size: {len(batch)}")
    print(f"First item type: {type(batch[0])}")
    print(f"First item is Datum: {isinstance(batch[0], Datum)}")
    break


Batch size: 4
First item type: <class 'tinkerbell.types.datum.Datum'>
First item is Datum: True


In [62]:
first_datum.to_torch()["model_input"]["input_ids"].shape

torch.Size([120])

In [63]:
batch[0].to_torch()["model_input"]["input_ids"].shape

torch.Size([706])